<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.2_Praktikum_Jenis_Machine_Learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum AI Modul 5.2: Jenis Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Capaian Pembelajaran Praktikum:
1. Mengimplementasikan **Supervised Learning** pada klasifikasi defisiensi hara kelapa sawit menggunakan data berlabel ($X, y$).
2. Menerapkan **Unsupervised Learning** untuk zonasi kesuburan tanah (K-Means), reduksi dimensi spektral (PCA), dan deteksi anomali sensor (Isolation Forest) pada data tanpa label.
3. Membangun simulasi **Reinforcement Learning** sederhana berbasis Markov Decision Process (MDP) dan algoritma Tabular Q-Learning untuk otomasi irigasi lahan gambut.
4. Membangun pipeline hibrida Semi-Supervised (PCA + Random Forest).

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-Learn tools untuk Supervised & Unsupervised Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, silhouette_score

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 1. Supervised Learning: Klasifikasi Defisiensi Hara Daun Sawit (Data Berlabel)
Pada bagian ini, model diberikan pasangan data input ($X$) berupa pengukuran laboratorium daun (Kadar N, P, K, Mg) dan label kebenaran dasar ($y$) berupa status kesehatan tanaman:
- Kelas 0: Tanaman Sehat / Nutrisi Seimbang
- Kelas 1: Defisiensi Nitrogen (Daun Menguning)
- Kelas 2: Defisiensi Magnesium (Bercak Jingga Klorosis)


In [ ]:
# 1. Bangun dataset berlabel (Supervised Dataset)
np.random.seed(42)
n_pohon = 600

# Fitur daun: [Kadar_N (%), Kadar_P (%), Kadar_K (%), Kadar_Mg (%)]
# Kelas 0: Sehat
n0 = np.random.normal(2.7, 0.2, 200)
p0 = np.random.normal(0.18, 0.02, 200)
k0 = np.random.normal(1.1, 0.15, 200)
mg0 = np.random.normal(0.35, 0.04, 200)
y0 = np.zeros(200, dtype=int)

# Kelas 1: Defisiensi Nitrogen (N drop)
n1 = np.random.normal(1.9, 0.2, 200)
p1 = np.random.normal(0.17, 0.02, 200)
k1 = np.random.normal(1.05, 0.15, 200)
mg1 = np.random.normal(0.34, 0.04, 200)
y1 = np.ones(200, dtype=int)

# Kelas 2: Defisiensi Magnesium (Mg drop)
n2 = np.random.normal(2.65, 0.2, 200)
p2 = np.random.normal(0.18, 0.02, 200)
k2 = np.random.normal(1.1, 0.15, 200)
mg2 = np.random.normal(0.18, 0.03, 200)
y2 = np.full(200, 2, dtype=int)

X_sup = np.vstack([
    np.column_stack([n0, p0, k0, mg0]),
    np.column_stack([n1, p1, k1, mg1]),
    np.column_stack([n2, p2, k2, mg2])
])
y_sup = np.concatenate([y0, y1, y2])

# Partisi Data Latih dan Uji
X_tr, X_te, y_tr, y_te = train_test_split(X_sup, y_sup, test_size=0.25, random_state=42, stratify=y_sup)

# Latih Model Klasifikasi Terarah (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_tr, y_tr)

# Evaluasi pada data uji
y_pred_sup = rf_model.predict(X_te)
print("=== HASIL EVALUASI SUPERVISED LEARNING (KLASIFIKASI HARA) ===")
print(classification_report(y_te, y_pred_sup, target_names=['Sehat (0)', 'Defisiensi N (1)', 'Defisiensi Mg (2)']))


## 2. Unsupervised Learning: Segmentasi Lahan (K-Means) & Reduksi Dimensi (PCA)
Pada bagian ini, kita mengolah data sensor tanah 500 blok kebun **tanpa label target**.
Tujuan:
1. **K-Means Clustering:** Mengelompokkan blok kebun ke dalam zona kesuburan alami.
2. **PCA:** Mereduksi dimensi fitur tanah dari 5 variabel menjadi 2 komponen utama untuk visualisasi.
3. **Isolation Forest:** Mendeteksi blok kebun dengan kondisi tanah anomali ekstrem (outlier).


In [ ]:
# 1. Dataset Tanpa Label (Unsupervised Data): 5 Fitur Sensor Tanah
np.random.seed(123)
n_blok = 500

# Variabel: [pH_Tanah, Kelembaban_%, C_Organik_%, Kapasitas_Tukar_Kation, Ketebalan_Gambut_cm]
zona_a = np.random.multivariate_normal([5.5, 45, 2.5, 18, 20], np.diag([0.2, 10, 0.3, 4, 15]), 180)
zona_b = np.random.multivariate_normal([4.2, 75, 4.8, 12, 120], np.diag([0.15, 8, 0.4, 3, 25]), 200)
zona_c = np.random.multivariate_normal([6.2, 35, 1.8, 22, 5], np.diag([0.25, 12, 0.2, 5, 5]), 120)

X_unsup = np.vstack([zona_a, zona_b, zona_c])
# Normalisasi fitur sebelum clustering dan PCA
scaler = StandardScaler()
X_unsup_scaled = scaler.fit_transform(X_unsup)

# 2. Klastering Mandiri Menggunakan K-Means (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_unsup_scaled)
sil_score = silhouette_score(X_unsup_scaled, cluster_labels)

# 3. Reduksi Dimensi Menggunakan PCA (5 Dimensi -> 2 Dimensi)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_unsup_scaled)

# 4. Deteksi Anomali Menggunakan Isolation Forest
iso = IsolationForest(contamination=0.03, random_state=42)
anomali_flags = iso.fit_predict(X_unsup_scaled) # -1 = Anomali, 1 = Normal

print("=== HASIL EVALUASI UNSUPERVISED LEARNING ===")
print(f"Silhouette Score K-Means (Kualitas Partisi Klaster): {sil_score:.4f}")
print(f"Rasio Variansi Terjelaskan PCA (2 Komponen):        {pca.explained_variance_ratio_.sum() * 100:.2f}%")
print(f"Jumlah Blok Anomali Terdeteksi (Isolation Forest):  {(anomali_flags == -1).sum()} blok")


## 3. Reinforcement Learning: Otomasi Pengendalian Irigasi Lahan Gambut
Pada paradigma **Reinforcement Learning**, agen cerdas belajar melalui interaksi dinamis dalam kerangka *Markov Decision Process* (MDP).
- **Status ($S$):** Level kelembaban gambut: Kering (0), Ideal (1), Basah/Banjir (2).
- **Aksi ($A$):** Tindakan irigasi: Diamkan (0), Nyalakan Pompa (1).
- **Fungsi Ganjaran ($R$):**
  - Bertahan di status Ideal: $+10$
  - Menjadi Kering (bahaya kebakaran): $-30$
  - Banjir / Genangan: $-15$


In [ ]:
# Simulasi Sederhana MDP Irigasi Lahan Gambut menggunakan Tabular Q-Learning
np.random.seed(42)

n_states = 3   # 0: Kering, 1: Ideal, 2: Banjir
n_actions = 2  # 0: Matikan Pompa, 1: Nyalakan Pompa

# Matriks Q(s, a) diinisialisasi dengan angka nol
Q_table = np.zeros((n_states, n_actions))

# Hiperparameter Q-Learning
alpha = 0.1       # Learning rate
gamma = 0.9       # Discount factor
epsilon = 0.2     # Exploration rate
n_episodes = 500  # Jumlah episode simulasi

def simulasi_lingkungan(state, action):
    # Dinamika transisi lingkungan kebun gambut
    if action == 1: # Nyalakan pompa -> kelembaban naik
        next_state = min(state + 1, 2)
    else:           # Matikan pompa -> gambut mengering secara alami
        next_state = max(state - 1, 0)
        
    # Fungsi Reward
    if next_state == 1:
        reward = 10.0   # Target optimal tercapai
    elif next_state == 0:
        reward = -30.0  # Bahaya gambut kering terbakar!
    else:
        reward = -15.0  # Genangan air berlebih
        
    return next_state, reward

# Loop Pembelajaran Penguatan (Q-Learning)
for ep in range(n_episodes):
    current_state = np.random.choice([0, 1, 2])
    
    for step in range(20):
        # Kebijakan Epsilon-Greedy (Eksplorasi vs Eksploitasi)
        if np.random.rand() < epsilon:
            action = np.random.choice(n_actions) # Eksplorasi acak
        else:
            action = np.argmax(Q_table[current_state]) # Eksploitasi aksi terbaik
            
        next_state, reward = simulasi_lingkungan(current_state, action)
        
        # Pembaruan Nilai Q via Persamaan Bellman
        best_future_q = np.max(Q_table[next_state])
        Q_table[current_state, action] += alpha * (
            reward + gamma * best_future_q - Q_table[current_state, action]
        )
        
        current_state = next_state

print("=== TABEL Q(s, a) HASIL OPTIMASI REINFORCEMENT LEARNING ===")
df_q = pd.DataFrame(
    Q_table.round(2),
    index=['Status 0: Kering', 'Status 1: Ideal', 'Status 2: Banjir'],
    columns=['Aksi 0: Matikan Pompa', 'Aksi 1: Nyalakan Pompa']
)
print(df_q)

kebijakan_optimal = [np.argmax(Q_table[s]) for s in range(n_states)]
aksi_map = {0: 'Matikan Pompa', 1: 'Nyalakan Pompa'}
print("\nKebijakan Keputusan Cerdas Agen Terlatih:")
for s, a in enumerate(kebijakan_optimal):
    print(f" - Saat kondisi {df_q.index[s]} -> Rekomendasi Agen: {aksi_map[a]}")


## 4. Tugas Tantangan Eksplorasi Mandiri (HOTS Challenge)

### Tantangan: Pipeline Hibrida Semi-Supervised (PCA + Random Forest)
Pada dataset daun sawit berdimensi banyak, terapkan teknik Unsupervised Learning (PCA) untuk mereduksi fitur masukan menjadi 2 komponen utama terlebih dahulu, lalu latih model Supervised Learning (Random Forest) di atas fitur terkompresi tersebut. Evaluasi apakah akurasi tetap terjaga tinggi!

### Solusi Tantangan:


In [ ]:
# 1. Terapkan Standarisasi dan PCA (Unsupervised) pada data fitur daun sawit
scaler_daun = StandardScaler()
X_tr_sc = scaler_daun.fit_transform(X_tr)
X_te_sc = scaler_daun.transform(X_te)

pca_daun = PCA(n_components=2)
X_tr_pca = pca_daun.fit_transform(X_tr_sc)
X_te_pca = pca_daun.transform(X_te_sc)

# 2. Latih Supervised Classifier pada representasi terkompresi
rf_hibrida = RandomForestClassifier(n_estimators=100, random_state=42)
rf_hibrida.fit(X_tr_pca, y_tr)

# 3. Evaluasi akurasi
skor_asli = rf_model.score(X_te, y_te)
skor_hibrida = rf_hibrida.score(X_te_pca, y_te)

print("=== EVALUASI PERFORMA PIPELINE HIBRIDA (UNSUPERVISED + SUPERVISED) ===")
print(f"Akurasi Model Asli (4 Fitur Laboratorium):       {skor_asli * 100:.2f}%")
print(f"Akurasi Model Hibrida (2 Komponen PCA Laten):   {skor_hibrida * 100:.2f}%")
print(f"Retensi Performa Akurasi:                        {(skor_hibrida / skor_asli) * 100:.2f}%")
print("Kesimpulan: Pipeline hibrida berhasil memangkas 50% dimensi fitur dengan tetap mempertahankan akurasi prima!")
